In [1]:
import numpy as np
import pandas as pd
import ast
import itertools
from tqdm import tqdm
import requests
from matplotlib import pyplot as plt
from PIL import Image, ImageOps
import streamlit as st
from st_clickable_images import clickable_images

st.set_page_config(layout="wide")

In [13]:
def createSession():
    if 'cards' not in st.session_state:
        st.session_state.cards = pd.read_csv("data/cards.csv")

    if 'defaultCard' not in st.session_state:
        st.session_state.defaultCard = pd.DataFrame({
            'id' : [0],
            'name' : ['Default'],
            'art' : ['https://marvelsnapzone.com/wp-content/themes/blocksy-child/assets/media/cards/annihilus.webp?v=20'],
            'ability' : [''],
            'slug' : [''],
        })

    if 'currentDeck' not in st.session_state:
        st.session_state.currentDeck = createEmptyDeck(st.session_state.defaultCard)
    
    if 'deck' not in st.session_state:
        st.session_state.decks = loadDecks()
        
def loadDecks():
    return pd.read_csv("data/decks.csv").iloc[:100]

def createEmptyDeck(defaultCard):
    deck = pd.DataFrame(columns=defaultCard.columns)
    for i in range(12):
        deck.loc[i] = defaultCard.loc[0]
    return deck

def addCard(card):
    firstDefaultCard = st.session_state.currentDeck[st.session_state.currentDeck.name == 'Default']
    if not firstDefaultCard.empty:
        st.session_state.currentDeck.iloc[firstDefaultCard.index[0]] = card
        
def drawCurrentDeck(deck, defaultCard):
    clicked = clickable_images(
        list(deck['art']),
        div_style={"display": "flex", "justify-content": "center", "flex-wrap": "wrap"},
        img_style={"margin": "5px", "height": "150px"},
    )

    if clicked > -1 and deck.iloc[clicked]['name'] != defaultCard.iloc[0]['name']:
        deck.iloc[clicked] = defaultCard.iloc[0]
        st.experimental_rerun()

def resetDeck(deck, defaultCard):
    for i in range(12):
        deck.loc[i] =  defaultCard.loc[0] 
    st.experimental_rerun()
    
def selectionDeck(decks):
    allDeckWithScore = []
    
    for i, deck in decks.iterrows():
        score = 0
        deckCards = [int(cardId) for cardId in deck['id_cards'][1:-1].split(',')]
        for j, card in st.session_state.currentDeck.loc[st.session_state.currentDeck['name'] != st.session_state.defaultCard.iloc[0]['name']].iterrows():
            if card['id'] in deckCards:
                score += 10
                
        allDeckWithScore.append({'score' : score, 'deck' : deck})
    
    topDecksByScore = sorted(allDeckWithScore, key=lambda d: d['score'], reverse=True)[:10]
    topDecks = [ topDeckByScore['deck'] for topDeckByScore in topDecksByScore ]
    
    return topDecks

def drawTopDecks(topDecks):
    for deck in topDecks:
        with st.container():
            st.header(deck['name'])
            images = []
            titles = []
            for cardId in deck['id_cards'][1:-1].split(','):
                card = st.session_state.cards[st.session_state.cards['id'] == int(cardId)].iloc[0]
                if int(cardId) in st.session_state.currentDeck['id'].values:
                    images.append("https://github.com/Fully-san/MarvelSnapDeckPrediction/blob/master/Cards/abomination.webp")#"https://cdn.filestackcontent.com/ANWxluLzPRk6VV28eF11Mz/monochrome/" + card['art'])
                else:
                    images.append(card['art'])
                titles.append(str(card['ability']))
                
            clicked = clickable_images(
                images,
                titles,
                div_style={"display": "flex", "justify-content": "center", "flex-wrap": "wrap"},
                img_style={"margin": "5px", "height": "200px"},
            )
            
            if(clicked > -1):
                cardId = deck['id_cards'][1:-1].split(',')[clicked]
                cardClicked = st.session_state.cards[st.session_state.cards['id'] == int(cardId)].iloc[0]
                st.header(cardClicked['name'])
                st.subheader(cardClicked['ability'])
            
            st.markdown("""---""")

In [12]:
createSession()

with st.sidebar:
    selectedCard = st.selectbox('', list(st.session_state.cards['name']))
    
    with st.container():
        c1, c2 = st.columns([3,1])
        
        with c1:
            if st.button('Add card'):
                addCard(st.session_state.cards.loc[st.session_state.cards['name'] == selectedCard].values)

        with c2:
            if st.button('Reset'):
                resetDeck(st.session_state.currentDeck, st.session_state.defaultCard)
        
    with st.container():
        drawCurrentDeck(st.session_state.currentDeck, st.session_state.defaultCard)
        
drawTopDecks(selectionDeck(st.session_state.decks))


2022-12-27 09:13:59.774 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2022-12-27 09:13:59.818 
  command:

    streamlit run c:\Users\0832825\Anaconda3\envs\StreamLit\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
